# Neural Collaborative Filtering avec PyTorch

Le réseau, le negative sampling, l'entraînement, l'évaluation et les recommandations sont définis dans `src/recommend_deep_learning.py`. Ce notebook montre uniquement le flux d'utilisation.

In [ ]:
import importlib
import sys
from pathlib import Path

import pandas as pd
import torch

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Impossible de trouver la racine MusicRecoSystem")
sys.path.insert(0, str(PROJECT_ROOT))

import src.recommend_deep_learning as ncf_module
ncf_module = importlib.reload(ncf_module)

## 1. Charger les données et choisir le matériel

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "listens.parquet"
listens = pd.read_parquet(DATA_PATH)
DEVICE = "auto"

pd.Series({
    "dataset_version": listens["dataset_version"].iloc[0],
    "interactions": len(listens),
    "users": listens["user"].nunique(),
    "artist_identities": listens["artist_uid"].nunique(),
    "mps_available": torch.backends.mps.is_available(),
    "cuda_available": torch.cuda.is_available(),
})

## 2. Sélectionner la configuration et entraîner le réseau final

Trois configurations d'embeddings et de régularisation sont comparées sur la validation. La meilleure est réentraînée sur train + validation.

In [ ]:
import src.recommend_deep_learning as ncf_module
ncf_module = importlib.reload(ncf_module)

experiment = ncf_module.select_and_fit_best_ncf(
    listens,
    epochs=5,
    top_n=10,
    random_state=42,
    device=DEVICE,
    verbose=True,
)

display(experiment.tuning_results)
display(pd.DataFrame({
    "validation": experiment.validation_metrics,
    "test_final": experiment.test_metrics,
}))
print(f"Dataset : {experiment.dataset_version}")
print(f"Meilleure configuration : {experiment.best_config}")

## 3. Vérifier la convergence du modèle final

In [ ]:
pd.DataFrame({
    "epoch": range(1, len(experiment.recommender.training_losses_) + 1),
    "training_loss": experiment.recommender.training_losses_,
})

## 4. Obtenir 10 recommandations

In [ ]:
USER = "BKILPATRICK11"

experiment.recommender.recommend(USER, top_n=10)